# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides an interactive guide for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"Dataset name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Version: {metadata.get('version', '(not specified)')}")
print(f"License: {metadata.get('license', '(not specified)')}")

## 2. Data Overview
Review available record sets, their fields, and all entity IDs using the Croissant schema.

We enumerate the record sets and the field `@id`s present in each.

In [ ]:
# Inspecting available record sets and fields by their `@id`
schema = dataset.metadata

# Each record set in Croissant is identified by its '@id' -- let's enumerate them
record_sets = []
if hasattr(schema, 'record_sets') and schema.record_sets:
    # For mlcroissant >=0.9 API
    for rs in schema.record_sets:
        rs_id = rs['@id']
        record_sets.append(rs_id)
        print(f"\nRecordSet @id: {rs_id}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        # Enumerate fields
        if 'field' in rs:
            print("  Fields (by @id):")
            for f in rs['field']:
                print(f"    - {f['@id']}  Name: {f.get('name', 'N/A')}")
else:
    # Some schemas store record sets directly in metadata['recordSet']
    record_sets = metadata.get('recordSet', [])
    if isinstance(record_sets, list) and len(record_sets) > 0:
        for rs in record_sets:
            rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
            print(f"RecordSet @id: {rs_id}")
    else:
        print("No record sets found in the schema.")

> 📝 **Note**: If no record sets are listed, the dataset's file may contain only a single main RecordSet. To proceed, you may need to consult the Croissant schema (open the JSON) to find the available records. The main survey data is usually in the first (or only) RecordSet.

In [ ]:
# Let's attempt to detect the main record set for this dataset

# Attempt to auto-detect the main record set @id
main_record_set_id = None
# 1. Check if metadata['recordSet'] is present and list-like
if 'recordSet' in metadata and isinstance(metadata['recordSet'], list) and len(metadata['recordSet']) > 0:
    first_rs = metadata['recordSet'][0]
    if isinstance(first_rs, dict) and '@id' in first_rs:
        main_record_set_id = first_rs['@id']
    elif isinstance(first_rs, str):
        main_record_set_id = first_rs

# 2. If not found, check in the old-style
if main_record_set_id is None:
    # Try to traverse the schema
    if hasattr(schema, 'record_sets') and schema.record_sets:
        rs = schema.record_sets[0]
        main_record_set_id = rs['@id']

print(f"Selected main RecordSet @id: {main_record_set_id}")

You can further inspect the first few records with their field `@id`s below:

In [ ]:
# View sample records from the selected RecordSet (by @id)
try:
    for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
        print(json.dumps(record, indent=2))
        if i >= 2:
            break
except Exception as e:
    print(f"Error fetching records: {e}\n\nCheck the schema structure and ensure the correct RecordSet @id is specified.")

## 3. Data Extraction
Load one or several record sets by their `@id` into pandas DataFrames for analysis. Field names (columns) are referenced by their `@id`. We'll extract all available record sets.

In [ ]:
# List of RecordSet @ids (can extend as needed)
record_set_ids = [main_record_set_id]  # If there are multiples, add more here

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded shape: {df.shape}")
        print(f"Available columns (@id):\n{df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for RecordSet {record_set_id}")

If you want to examine the names and descriptions of the fields, use the Croissant schema and match fields by their `@id`.

The remainder of this notebook assumes the DataFrame for the main record set (converted above) is in `dataframes[main_record_set_id]`.

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps—filtering, normalization, grouping—referenced by Croissant field `@id`s.

We'll:
1. Select a numeric field (e.g., GAD-7 total score, referenced by its Croissant `@id`).
2. Filter records where this score exceeds a threshold.
3. Normalize the values of this field.
4. Group by another (categorical) field such as age group or gender, specified by its `@id`.

In [ ]:
# Example field @ids based on the dataset (consult JSON or dataframes columns):
# Let's suppose the GAD-7 total score column is '@gad_7_total_score' and gender is '@gender'.

numeric_field_id = '@gad_7_total_score'   # CHANGE THIS to match the true @id of GAD-7 total score
group_field_id = '@gender'                # CHANGE THIS to match true @id of gender field
record_set_id = main_record_set_id

df = dataframes[record_set_id]

# If @id uses something else (consult previous printouts), set these to the actual @id
if numeric_field_id not in df.columns:
    print("The suggested numeric_field_id (@gad_7_total_score) was not found in DataFrame columns.\nColumns are:", df.columns.tolist())
else:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped average {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

> **Tip**: If the suggested field `@id`s do not match, consult your DataFrame columns above, or refer to the schema file for the correct field names.

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of GAD-7 total scores (field by @id)
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field_id} (GAD-7 Total Score)')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Boxplot by gender/categorical group
if numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(7,3))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()
else:
    print("Either numeric_field_id or group_field_id not found in DataFrame columns for visualization.")

## 6. Conclusion
This notebook demonstrated loading, inspecting, processing, and visualizing the FAIR² Mental Health Survey dataset from Kilifi County, Kenya using Croissant `@id`-centric best practices.

**Key Observations:**
- Data can be cleanly referenced and filtered using schema `@id`s.
- Simple group and normalization operations can be composed on survey scores (e.g., GAD-7) and categorical fields (e.g., gender).
- The `mlcroissant` library enables standards-compliant data extraction that is compatible with AI-ready pipelines.

Continue your exploration by referencing other RecordSets or fields, or integrating this workflow into larger ML or data science analyses!